# Autoencoder-Based Face Recognition on LFW — Final Script

**Project A3010-251 — Autoencoder for Face Recognition**

This notebook implements a **joint reconstruction + recognition** pipeline:

1. **Phase 1 — Joint Training:** Train a deep convolutional autoencoder with a **dual loss**: reconstruction (SSIM+MSE) on ALL of LFW + classification (cross-entropy) on labelled identities. This forces the latent space to be both reconstructive AND identity-discriminative from the start.
2. **Phase 2 — Fine-Tuning:** End-to-end fine-tuning with discriminative learning rates for maximum classification accuracy.

The Phase 2 data pipeline, identity selection, and splits are **identical** to the VGG-based classifier notebook to ensure a fair comparative study.

### Key Changes from Previous Version
- **Joint reconstruction + classification loss in Phase 1** — the latent space learns identity-discriminative features from the start, not just reconstruction features
- **Longer fine-tuning** with higher base learning rate and more epochs
- **TF 2.19 compatibility** — fixed `var.ref()` deprecation
- **Comprehensive saving** — all models, histories, metrics, and plots saved for report

**Key evaluation principle:** The test set is never used during training or model selection. Final reported numbers come exclusively from the held-out test set.

---
## 0 — Imports & Setup

In [ ]:
import os, glob, random, json, time
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, backend as K
from tensorflow.keras.models import Model
from collections import Counter
import matplotlib.pyplot as plt

print("TensorFlow version:", tf.__version__)
print("GPU available:", tf.config.list_physical_devices('GPU'))

---
## 1 — Configuration

Values marked `MUST MATCH VGG` are shared with the VGG notebook to guarantee identical data.

In [ ]:
# == Paths ==
DATA_DIR = "/kaggle/input/datasets/jessicali9530/lfw-dataset/lfw-deepfunneled/lfw-deepfunneled"   # MUST MATCH VGG

OUT_DIR = "/kaggle/working/ae_classifier"
os.makedirs(OUT_DIR, exist_ok=True)

# == Shared data params (MUST MATCH VGG) ==
IMG_SIZE          = 128       # MUST MATCH VGG
BATCH_SIZE        = 64        # MUST MATCH VGG
SEED              = 42        # MUST MATCH VGG
MIN_IMAGES_PER_ID = 10        # MUST MATCH VGG
TOP_K             = 200       # MUST MATCH VGG
TRAIN_RATIO       = 0.70      # MUST MATCH VGG
VAL_RATIO         = 0.15      # MUST MATCH VGG

# == Autoencoder-specific ==
LATENT_DIM          = 512
AE_EPOCHS           = 80       # Phase 1: joint reconstruction + classification
CLF_EPOCHS_FROZEN   = 40       # Phase 2a: classifier head only (encoder frozen)
CLF_EPOCHS_FINETUNE = 60       # Phase 2b: full fine-tune (increased from 30)
AE_LR               = 1e-3
CLF_LR_FROZEN       = 3e-4
CLF_LR_FINETUNE     = 5e-5     # Increased from 1e-5

# == Joint loss weighting ==
RECON_WEIGHT = 1.0    # reconstruction loss weight
CLF_WEIGHT   = 0.5    # classification loss weight (Phase 1 joint training)

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("Config OK")

---
## 2 — Data Preparation

### 2a: Classification Split (identical to VGG notebook)

We prepare this FIRST because we need the identity labels for the joint training in Phase 1.

In [ ]:
# == Identity filtering (MUST MATCH VGG) ==
random.seed(SEED)
np.random.seed(SEED)

people = []
for person in os.listdir(DATA_DIR):
    pdir = os.path.join(DATA_DIR, person)
    if not os.path.isdir(pdir):
        continue
    imgs = glob.glob(os.path.join(pdir, "*.jpg")) + glob.glob(os.path.join(pdir, "*.png"))
    if len(imgs) >= MIN_IMAGES_PER_ID:
        people.append((person, len(imgs)))

people.sort(key=lambda x: x[1], reverse=True)
people = people[:TOP_K]

class_names = [p for p, _ in people]
class_to_idx = {c: i for i, c in enumerate(class_names)}
NUM_CLASSES = len(class_names)

print(f"Classes kept: {NUM_CLASSES}")
print(f"Top 5: {people[:5]}")

In [ ]:
# == 3-way split: 70/15/15 per identity (MUST MATCH VGG) ==
train_paths, train_labels = [], []
val_paths,   val_labels   = [], []
test_paths,  test_labels  = [], []

for person, _ in people:
    pdir = os.path.join(DATA_DIR, person)
    imgs = sorted(glob.glob(os.path.join(pdir, "*.jpg")) + glob.glob(os.path.join(pdir, "*.png")))
    random.shuffle(imgs)

    n = len(imgs)
    n_train = int(TRAIN_RATIO * n)
    n_val   = int(VAL_RATIO * n)
    n_train = max(1, n_train)
    n_val   = max(1, n_val)
    if n_train + n_val >= n:
        n_train = n - n_val - 1
        n_train = max(1, n_train)
    n_test = n - n_train - n_val
    if n_test < 1:
        n_train -= 1
        n_test = 1

    tr = imgs[:n_train]
    va = imgs[n_train:n_train + n_val]
    te = imgs[n_train + n_val:]

    label = class_to_idx[person]
    train_paths.extend(tr);  train_labels.extend([label] * len(tr))
    val_paths.extend(va);    val_labels.extend([label]   * len(va))
    test_paths.extend(te);   test_labels.extend([label]  * len(te))

print(f"Train: {len(train_paths)}   Val: {len(val_paths)}   Test: {len(test_paths)}")
print(f"Min train/class: {min(Counter(train_labels).values())}")
print(f"Min val/class:   {min(Counter(val_labels).values())}")
print(f"Min test/class:  {min(Counter(test_labels).values())}")

with open(os.path.join(OUT_DIR, "class_names.json"), "w") as f:
    json.dump(class_names, f, indent=2)

### 2b: Phase 1 Data — ALL of LFW with Identity Labels

For joint training, we need images paired with identity labels.
- Images from known identities (in our classification set) get their class label.
- Images from unknown identities get label `-1` (excluded from classification loss, still used for reconstruction).

In [ ]:
# == Collect ALL LFW images with identity labels ==
random.seed(SEED)

all_people_dirs = sorted([
    d for d in os.listdir(DATA_DIR)
    if os.path.isdir(os.path.join(DATA_DIR, d))
])

MAX_PER_PERSON = 20

ae_all_paths = []
ae_all_labels = []  # -1 for unknown identities

known_count = 0
unknown_count = 0

for person in all_people_dirs:
    pdir = os.path.join(DATA_DIR, person)
    imgs = sorted(glob.glob(os.path.join(pdir, "*.jpg")) + glob.glob(os.path.join(pdir, "*.png")))
    if len(imgs) > 0:
        random.shuffle(imgs)
        selected = imgs[:MAX_PER_PERSON]
        ae_all_paths.extend(selected)
        if person in class_to_idx:
            ae_all_labels.extend([class_to_idx[person]] * len(selected))
            known_count += len(selected)
        else:
            ae_all_labels.extend([-1] * len(selected))
            unknown_count += len(selected)

# Shuffle together
combined = list(zip(ae_all_paths, ae_all_labels))
random.shuffle(combined)
ae_all_paths, ae_all_labels = zip(*combined)
ae_all_paths = list(ae_all_paths)
ae_all_labels = list(ae_all_labels)

# 80/20 split for AE train/val
ae_split = int(0.8 * len(ae_all_paths))
ae_train_paths  = ae_all_paths[:ae_split]
ae_train_labels = ae_all_labels[:ae_split]
ae_val_paths    = ae_all_paths[ae_split:]
ae_val_labels   = ae_all_labels[ae_split:]

print(f"Total identities in LFW: {len(all_people_dirs)}")
print(f"Total images selected (max {MAX_PER_PERSON}/person): {len(ae_all_paths)}")
print(f"  Known identities: {known_count}  Unknown: {unknown_count}")
print(f"AE train: {len(ae_train_paths)}   AE val: {len(ae_val_paths)}")

In [ ]:
# == tf.data datasets ==
AUTOTUNE = tf.data.AUTOTUNE

def load_image(fp):
    img = tf.io.read_file(fp)
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.image.resize(img, (IMG_SIZE, IMG_SIZE))
    img = tf.cast(img, tf.float32) / 255.0
    return img

def augment(img):
    img = tf.image.random_flip_left_right(img)
    img = tf.image.random_brightness(img, 0.15)
    img = tf.image.random_contrast(img, 0.85, 1.15)
    img = tf.image.random_saturation(img, 0.85, 1.15)
    crop_size = tf.random.uniform([], minval=int(IMG_SIZE * 0.85), maxval=IMG_SIZE + 1, dtype=tf.int32)
    img = tf.image.random_crop(img, [crop_size, crop_size, 3])
    img = tf.image.resize(img, (IMG_SIZE, IMG_SIZE))
    img = tf.clip_by_value(img, 0.0, 1.0)
    return img

# Phase 1 datasets: (image, label) — label is -1 for unknown
def load_ae_train(fp, label):
    img = load_image(fp)
    img = augment(img)
    return img, label

def load_ae_val(fp, label):
    img = load_image(fp)
    return img, label

train_ds_ae = (
    tf.data.Dataset.from_tensor_slices((ae_train_paths, ae_train_labels))
    .shuffle(len(ae_train_paths), seed=SEED)
    .map(load_ae_train, num_parallel_calls=AUTOTUNE)
    .batch(BATCH_SIZE).prefetch(AUTOTUNE)
)
val_ds_ae = (
    tf.data.Dataset.from_tensor_slices((ae_val_paths, ae_val_labels))
    .map(load_ae_val, num_parallel_calls=AUTOTUNE)
    .batch(BATCH_SIZE).prefetch(AUTOTUNE)
)

# Phase 2 classification datasets
def load_clf_train(fp, y):
    img = load_image(fp)
    img = augment(img)
    return img, y

def load_clf_val(fp, y):
    img = load_image(fp)
    return img, y

train_ds_clf = (
    tf.data.Dataset.from_tensor_slices((train_paths, train_labels))
    .shuffle(2000, seed=SEED)
    .map(load_clf_train, num_parallel_calls=AUTOTUNE)
    .batch(BATCH_SIZE).prefetch(AUTOTUNE)
)
val_ds_clf = (
    tf.data.Dataset.from_tensor_slices((val_paths, val_labels))
    .map(load_clf_val, num_parallel_calls=AUTOTUNE)
    .batch(BATCH_SIZE).prefetch(AUTOTUNE)
)
test_ds_clf = (
    tf.data.Dataset.from_tensor_slices((test_paths, test_labels))
    .map(load_clf_val, num_parallel_calls=AUTOTUNE)
    .batch(BATCH_SIZE).prefetch(AUTOTUNE)
)

print("All datasets ready.")

---
## 3 — Model Architecture

Same VGG-style encoder as before, but now we build a **joint model** with two output heads:
- **Reconstruction head** (decoder): outputs reconstructed image
- **Classification head**: outputs class probabilities from the latent space

During Phase 1, both heads are trained simultaneously with a weighted loss.

In [ ]:
# == Custom SSIM + MSE Loss ==

def ssim_mse_loss(y_true, y_pred, alpha=0.7):
    ssim_val = tf.reduce_mean(tf.image.ssim(y_true, y_pred, max_val=1.0))
    mse_val  = tf.reduce_mean(tf.square(y_true - y_pred))
    return alpha * (1.0 - ssim_val) + (1.0 - alpha) * mse_val

In [ ]:
def enc_block(x, filters, n_convs, block_name, use_batch_norm=True):
    """VGG-style encoder block: n_convs x Conv2D + BN + ReLU, then MaxPool."""
    for i in range(n_convs):
        x = layers.Conv2D(filters, (3, 3), padding="same",
                          kernel_initializer="he_normal",
                          name=f"{block_name}_conv{i+1}")(x)
        if use_batch_norm:
            x = layers.BatchNormalization(name=f"{block_name}_bn{i+1}")(x)
        x = layers.Activation("relu", name=f"{block_name}_relu{i+1}")(x)
    x = layers.MaxPooling2D((2, 2), strides=2, name=f"{block_name}_pool")(x)
    return x


def dec_block(x, filters, n_convs, block_name, use_batch_norm=True, final_block=False):
    """Decoder block: UpSampling + n_convs x Conv2D + BN + ReLU."""
    x = layers.UpSampling2D((2, 2), name=f"{block_name}_upsample")(x)
    for i in range(n_convs):
        is_last_conv = final_block and (i == n_convs - 1)
        x = layers.Conv2D(filters if not is_last_conv else 3, (3, 3), padding="same",
                          kernel_initializer="he_normal",
                          name=f"{block_name}_conv{i+1}")(x)
        if is_last_conv:
            x = layers.Activation("sigmoid", name="dec_sigmoid")(x)
        else:
            if use_batch_norm:
                x = layers.BatchNormalization(name=f"{block_name}_bn{i+1}")(x)
            x = layers.Activation("relu", name=f"{block_name}_relu{i+1}")(x)
    return x


def build_models(input_shape, latent_dim, num_classes):
    """
    Build encoder, decoder, autoencoder, and joint model.
    Joint model has two outputs: [reconstruction, classification].
    """
    inp = layers.Input(shape=input_shape, name="encoder_input")

    # == Encoder (VGG-style) ==
    x = enc_block(inp, 64,  2, "enc_b1")   # 128 -> 64
    x = enc_block(x,  128, 2, "enc_b2")    # 64  -> 32
    x = enc_block(x,  256, 3, "enc_b3")    # 32  -> 16
    x = enc_block(x,  512, 3, "enc_b4")    # 16  -> 8
    x = enc_block(x,  512, 3, "enc_b5")    # 8   -> 4

    # == Bottleneck ==
    x = layers.GlobalAveragePooling2D(name="enc_gap")(x)   # (batch, 512)
    z = layers.Dense(latent_dim, name="latent_embedding")(x)
    z = layers.BatchNormalization(name="latent_bn")(z)
    z = layers.Activation("relu", name="latent_relu")(z)

    encoder = Model(inp, z, name="Encoder")

    # == Decoder ==
    x_dec = layers.Dense(4 * 4 * 512, name="dec_dense")(z)
    x_dec = layers.Reshape((4, 4, 512), name="dec_reshape")(x_dec)
    x_dec = dec_block(x_dec, 512, 3, "dec_b5")    # 4  -> 8
    x_dec = dec_block(x_dec, 256, 3, "dec_b4")     # 8  -> 16
    x_dec = dec_block(x_dec, 128, 3, "dec_b3")     # 16 -> 32
    x_dec = dec_block(x_dec, 64,  2, "dec_b2")     # 32 -> 64
    x_dec = dec_block(x_dec, 32,  2, "dec_b1", final_block=True)  # 64 -> 128

    autoencoder = Model(inp, x_dec, name="Face_Autoencoder")

    # == Classification head (from latent space) ==
    x_clf = layers.Dense(512, kernel_initializer="he_normal", name="joint_clf_fc1")(z)
    x_clf = layers.BatchNormalization(name="joint_clf_bn1")(x_clf)
    x_clf = layers.Activation("relu", name="joint_clf_relu1")(x_clf)
    x_clf = layers.Dropout(0.5, name="joint_clf_drop1")(x_clf)
    x_clf = layers.Dense(256, kernel_initializer="he_normal", name="joint_clf_fc2")(x_clf)
    x_clf = layers.BatchNormalization(name="joint_clf_bn2")(x_clf)
    x_clf = layers.Activation("relu", name="joint_clf_relu2")(x_clf)
    x_clf = layers.Dropout(0.3, name="joint_clf_drop2")(x_clf)
    clf_out = layers.Dense(num_classes, activation="softmax", name="joint_clf_softmax")(x_clf)

    # Joint model: input -> [reconstruction, classification]
    joint_model = Model(inp, [x_dec, clf_out], name="Joint_AE_Classifier")

    return encoder, autoencoder, joint_model


encoder, autoencoder, joint_model = build_models(
    input_shape=(IMG_SIZE, IMG_SIZE, 3),
    latent_dim=LATENT_DIM,
    num_classes=NUM_CLASSES,
)

print("\n" + "="*60)
print("ENCODER SUMMARY")
print("="*60)
encoder.summary()

print("\n" + "="*60)
print("JOINT MODEL SUMMARY")
print("="*60)
joint_model.summary()

In [ ]:
# == Model Size ==
def count_params(model):
    trainable     = int(np.sum([K.count_params(w) for w in model.trainable_weights]))
    non_trainable = int(np.sum([K.count_params(w) for w in model.non_trainable_weights]))
    return trainable, non_trainable

ae_train_p, ae_nontrain_p = count_params(autoencoder)
enc_train_p, enc_nontrain_p = count_params(encoder)
joint_train_p, joint_nontrain_p = count_params(joint_model)

print(f"Autoencoder  — trainable: {ae_train_p:,}   non-trainable: {ae_nontrain_p:,}   total: {ae_train_p+ae_nontrain_p:,}")
print(f"Encoder only — trainable: {enc_train_p:,}   non-trainable: {enc_nontrain_p:,}   total: {enc_train_p+enc_nontrain_p:,}")
print(f"Joint model  — trainable: {joint_train_p:,}   non-trainable: {joint_nontrain_p:,}   total: {joint_train_p+joint_nontrain_p:,}")
print(f"Estimated Encoder model size: {(enc_train_p + enc_nontrain_p) * 4 / 1e6:.2f} MB (float32)")

---
## 4 — Phase 1: Joint Reconstruction + Classification Training

This is the critical change. Instead of training reconstruction-only, we train with:

**Total Loss = RECON_WEIGHT * SSIM_MSE(input, reconstruction) + CLF_WEIGHT * CrossEntropy(label, prediction)**

The classification loss is only computed for images with known identity labels (not -1).
This forces the latent space to encode identity-discriminative features from the very start.

In [ ]:
optimizer_phase1 = keras.optimizers.Adam(learning_rate=AE_LR)
clf_loss_fn = keras.losses.SparseCategoricalCrossentropy(from_logits=False)

@tf.function
def train_step_joint(images, labels):
    with tf.GradientTape() as tape:
        recon, clf_probs = joint_model(images, training=True)

        # Reconstruction loss (all images)
        recon_loss = ssim_mse_loss(images, recon)

        # Classification loss (only known identities, label >= 0)
        # Use masking instead of tf.cond to avoid AutoGraph variable issues
        known_mask = tf.cast(labels >= 0, tf.float32)  # 1.0 for known, 0.0 for unknown
        safe_labels = tf.maximum(labels, 0)  # replace -1 with 0 (won't affect loss due to mask)
        per_sample_loss = keras.losses.sparse_categorical_crossentropy(safe_labels, clf_probs)
        # Mask: only count known identities, avoid division by zero
        n_known = tf.maximum(tf.reduce_sum(known_mask), 1.0)
        clf_loss = tf.reduce_sum(per_sample_loss * known_mask) / n_known

        total_loss = RECON_WEIGHT * recon_loss + CLF_WEIGHT * clf_loss

    grads = tape.gradient(total_loss, joint_model.trainable_variables)
    optimizer_phase1.apply_gradients(zip(grads, joint_model.trainable_variables))

    # Classification accuracy (on known identities only)
    clf_preds = tf.argmax(clf_probs, axis=1)
    correct = tf.cast(clf_preds == tf.cast(safe_labels, tf.int64), tf.float32)
    clf_acc = tf.reduce_sum(correct * known_mask) / n_known

    return total_loss, recon_loss, clf_loss, clf_acc

@tf.function
def val_step_joint(images, labels):
    recon, clf_probs = joint_model(images, training=False)
    recon_loss = ssim_mse_loss(images, recon)

    known_mask = tf.cast(labels >= 0, tf.float32)
    safe_labels = tf.maximum(labels, 0)
    per_sample_loss = keras.losses.sparse_categorical_crossentropy(safe_labels, clf_probs)
    n_known = tf.maximum(tf.reduce_sum(known_mask), 1.0)
    clf_loss = tf.reduce_sum(per_sample_loss * known_mask) / n_known

    clf_preds = tf.argmax(clf_probs, axis=1)
    correct = tf.cast(clf_preds == tf.cast(safe_labels, tf.int64), tf.float32)
    clf_acc = tf.reduce_sum(correct * known_mask) / n_known

    total_loss = RECON_WEIGHT * recon_loss + CLF_WEIGHT * clf_loss
    return total_loss, recon_loss, clf_loss, clf_acc

print("Joint training functions compiled.")

In [ ]:
# == Phase 1: Joint Training Loop ==
print(f"Phase 1: Joint training on {len(ae_train_paths)} images ({len(ae_val_paths)} val)")
print(f"  Reconstruction weight: {RECON_WEIGHT}  Classification weight: {CLF_WEIGHT}")
print(f"  Epochs: {AE_EPOCHS}  LR: {AE_LR}")
print()

ae_best_path = os.path.join(OUT_DIR, "joint_best.weights.h5")

ae_start_time = time.time()
best_val_loss = float('inf')
patience_counter = 0
PATIENCE_P1 = 10
lr_wait = 0
LR_PATIENCE_P1 = 4
prev_val_loss = float('inf')
current_lr = AE_LR

phase1_history = {
    "total_loss": [], "recon_loss": [], "clf_loss": [], "clf_acc": [],
    "val_total_loss": [], "val_recon_loss": [], "val_clf_loss": [], "val_clf_acc": []
}

for epoch in range(AE_EPOCHS):
    # Training
    t_total, t_recon, t_clf, t_acc = [], [], [], []
    for batch_imgs, batch_labels in train_ds_ae:
        tl, rl, cl, ca = train_step_joint(batch_imgs, batch_labels)
        t_total.append(float(tl)); t_recon.append(float(rl))
        t_clf.append(float(cl)); t_acc.append(float(ca))

    # Validation
    v_total, v_recon, v_clf, v_acc = [], [], [], []
    for batch_imgs, batch_labels in val_ds_ae:
        tl, rl, cl, ca = val_step_joint(batch_imgs, batch_labels)
        v_total.append(float(tl)); v_recon.append(float(rl))
        v_clf.append(float(cl)); v_acc.append(float(ca))

    # Average metrics
    tr_total, tr_recon = np.mean(t_total), np.mean(t_recon)
    tr_clf, tr_acc = np.mean(t_clf), np.mean(t_acc)
    vl_total, vl_recon = np.mean(v_total), np.mean(v_recon)
    vl_clf, vl_acc = np.mean(v_clf), np.mean(v_acc)

    phase1_history["total_loss"].append(tr_total)
    phase1_history["recon_loss"].append(tr_recon)
    phase1_history["clf_loss"].append(tr_clf)
    phase1_history["clf_acc"].append(tr_acc)
    phase1_history["val_total_loss"].append(vl_total)
    phase1_history["val_recon_loss"].append(vl_recon)
    phase1_history["val_clf_loss"].append(vl_clf)
    phase1_history["val_clf_acc"].append(vl_acc)

    # Checkpoint on val total loss
    improved = ""
    if vl_total < best_val_loss:
        best_val_loss = vl_total
        joint_model.save_weights(ae_best_path)
        patience_counter = 0
        improved = " << saved"
    else:
        patience_counter += 1

    # LR reduction
    if vl_total >= prev_val_loss:
        lr_wait += 1
        if lr_wait >= LR_PATIENCE_P1:
            current_lr *= 0.5
            current_lr = max(current_lr, 1e-6)
            optimizer_phase1.learning_rate.assign(current_lr)
            lr_wait = 0
            print(f"  >> LR reduced to {current_lr:.1e}")
    else:
        lr_wait = 0
    prev_val_loss = vl_total

    print(f"Epoch {epoch+1:2d}/{AE_EPOCHS} — loss: {tr_total:.4f} (recon: {tr_recon:.4f}, clf: {tr_clf:.4f}, acc: {tr_acc:.3f}) "
          f"| val_loss: {vl_total:.4f} (recon: {vl_recon:.4f}, clf: {vl_clf:.4f}, acc: {vl_acc:.3f}){improved}")

    if patience_counter >= PATIENCE_P1:
        print(f"\nEarly stopping at epoch {epoch+1}")
        break

# Load best weights
joint_model.load_weights(ae_best_path)
ae_train_time = time.time() - ae_start_time
print(f"\nPhase 1 training time: {ae_train_time:.1f}s ({ae_train_time/60:.1f} min)")
print(f"Best val loss: {best_val_loss:.4f}")

# Save history
with open(os.path.join(OUT_DIR, "phase1_history.json"), "w") as f:
    json.dump(phase1_history, f, indent=2)

### 4b: Phase 1 — Training Curves

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].plot(phase1_history["recon_loss"], label="Train Recon")
axes[0].plot(phase1_history["val_recon_loss"], label="Val Recon")
axes[0].set_title("Phase 1 — Reconstruction Loss (SSIM+MSE)")
axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Loss")
axes[0].legend(); axes[0].grid(True)

axes[1].plot(phase1_history["clf_loss"], label="Train Clf Loss")
axes[1].plot(phase1_history["val_clf_loss"], label="Val Clf Loss")
axes[1].set_title("Phase 1 — Classification Loss")
axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("Loss")
axes[1].legend(); axes[1].grid(True)

axes[2].plot(phase1_history["clf_acc"], label="Train Clf Acc")
axes[2].plot(phase1_history["val_clf_acc"], label="Val Clf Acc")
axes[2].set_title("Phase 1 — Classification Accuracy")
axes[2].set_xlabel("Epoch"); axes[2].set_ylabel("Accuracy")
axes[2].legend(); axes[2].grid(True)

plt.suptitle("Phase 1 — Joint Reconstruction + Classification Training", fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "phase1_curves.png"), dpi=150)
plt.show()

### 4c: Phase 1 — Reconstruction Samples

In [ ]:
for batch_imgs, _ in val_ds_ae.take(1):
    sample_imgs = batch_imgs

recon, _ = joint_model.predict(sample_imgs, verbose=0)

n = min(10, sample_imgs.shape[0])
fig, axes = plt.subplots(2, n, figsize=(2.5*n, 5))
for i in range(n):
    axes[0, i].imshow(sample_imgs[i].numpy())
    axes[0, i].set_title("Original", fontsize=8)
    axes[0, i].axis("off")
    axes[1, i].imshow(np.clip(recon[i], 0, 1))
    axes[1, i].set_title("Reconstructed", fontsize=8)
    axes[1, i].axis("off")

plt.suptitle(f"Autoencoder Reconstruction (latent_dim={LATENT_DIM})", fontsize=12)
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "ae_reconstruction_samples.png"), dpi=150)
plt.show()

---
## 5 — Raw Embedding Quality Check (KNN on Encoder)

Test whether the joint-trained encoder embeddings are already discriminative BEFORE Phase 2.

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

print("Extracting train embeddings...")
train_embeddings = encoder.predict(train_ds_clf, verbose=1)
print("Extracting test embeddings...")
test_embeddings  = encoder.predict(test_ds_clf, verbose=1)

train_labels_arr = np.array(train_labels)
test_labels_arr  = np.array(test_labels)

knn_results_pretrained = {}
for k in [1, 3, 5, 7]:
    knn = KNeighborsClassifier(n_neighbors=k, metric="cosine", n_jobs=-1)
    knn.fit(train_embeddings, train_labels_arr)
    knn_preds = knn.predict(test_embeddings)
    knn_acc = accuracy_score(test_labels_arr, knn_preds)
    knn_results_pretrained[k] = knn_acc
    print(f"KNN (k={k}, cosine) test accuracy: {knn_acc*100:.2f}%")

best_knn_k = max(knn_results_pretrained, key=knn_results_pretrained.get)
best_knn_acc = knn_results_pretrained[best_knn_k]
print(f"\nBest KNN (pretrained): k={best_knn_k} -> test accuracy: {best_knn_acc*100:.2f}%")

---
## 6 — Phase 2a: Classification on FROZEN Encoder

Freeze encoder, train a fresh classification head on top.

In [ ]:
# Freeze encoder
for layer in encoder.layers:
    layer.trainable = False

# Build classifier on encoder
def build_ae_classifier(encoder, num_classes):
    inp = encoder.input
    z   = encoder.output
    x   = layers.Dense(512, kernel_initializer="he_normal", name="clf_fc1")(z)
    x   = layers.BatchNormalization(name="clf_bn1")(x)
    x   = layers.Activation("relu", name="clf_relu1")(x)
    x   = layers.Dropout(0.5, name="clf_drop1")(x)
    x   = layers.Dense(256, kernel_initializer="he_normal", name="clf_fc2")(x)
    x   = layers.BatchNormalization(name="clf_bn2")(x)
    x   = layers.Activation("relu", name="clf_relu2")(x)
    x   = layers.Dropout(0.3, name="clf_drop2")(x)
    out = layers.Dense(num_classes, activation="softmax", name="clf_softmax")(x)
    return Model(inp, out, name="AE_Classifier")

clf_model = build_ae_classifier(encoder, NUM_CLASSES)

clf_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=CLF_LR_FROZEN),
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=False),
    metrics=["accuracy"],
)

trainable_count = sum(1 for l in clf_model.layers if l.trainable and len(l.weights) > 0)
frozen_count = sum(1 for l in clf_model.layers if not l.trainable and len(l.weights) > 0)
print(f"Trainable layers: {trainable_count}   Frozen layers: {frozen_count}")
clf_model.summary()

In [ ]:
clf_best_path_frozen = os.path.join(OUT_DIR, "clf_frozen_best.weights.h5")

clf_callbacks_frozen = [
    keras.callbacks.ModelCheckpoint(
        clf_best_path_frozen, monitor="val_accuracy", mode="max",
        save_best_only=True, save_weights_only=True, verbose=1
    ),
    keras.callbacks.EarlyStopping(
        monitor="val_accuracy", mode="max", patience=10,
        restore_best_weights=True, verbose=1
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss", factor=0.5, patience=3, verbose=1
    ),
]

print("Phase 2a: Training classifier HEAD ONLY (encoder frozen)...\n")
clf_frozen_start = time.time()

clf_history_frozen = clf_model.fit(
    train_ds_clf,
    validation_data=val_ds_clf,
    epochs=CLF_EPOCHS_FROZEN,
    callbacks=clf_callbacks_frozen,
)

clf_frozen_time = time.time() - clf_frozen_start
print(f"\nPhase 2a training time: {clf_frozen_time:.1f}s ({clf_frozen_time/60:.1f} min)")
print(f"Best val accuracy: {max(clf_history_frozen.history['val_accuracy']):.4f}")

---
## 7 — Phase 2b: Fine-Tune ALL Encoder Blocks (Discriminative LR)

Unfreeze the entire encoder and fine-tune end-to-end with discriminative learning rates.
Higher base LR than before (5e-5 vs 1e-5) and more epochs (60 vs 30).

In [ ]:
clf_model.load_weights(clf_best_path_frozen)

# Unfreeze all
for layer in clf_model.layers:
    layer.trainable = True

def get_lr_multiplier(layer_name):
    if "enc_b1" in layer_name or "enc_b2" in layer_name:
        return 0.01
    elif "enc_b3" in layer_name:
        return 0.1
    elif "enc_b4" in layer_name:
        return 0.5
    elif "enc_b5" in layer_name or "enc_gap" in layer_name or "latent_" in layer_name:
        return 1.0
    elif "clf_" in layer_name:
        return 5.0
    else:
        return 1.0

# Use id() instead of .ref() for TF 2.19 compatibility
var_multipliers = {}
for var in clf_model.trainable_variables:
    var_multipliers[id(var)] = get_lr_multiplier(var.name)

base_lr = CLF_LR_FINETUNE
print("Discriminative learning rate assignments:")
print(f"  Base LR: {base_lr}")
printed_groups = set()
for layer in clf_model.layers:
    if len(layer.weights) > 0 and layer.trainable:
        mult = get_lr_multiplier(layer.name)
        group = "_".join(layer.name.split("_")[:2]) if "_" in layer.name else layer.name
        if group not in printed_groups:
            printed_groups.add(group)
            print(f"  {layer.name:30s} -> mult={mult:.2f}  effective_lr={base_lr*mult:.1e}")

In [ ]:
optimizer_ft = keras.optimizers.Adam(learning_rate=base_lr)
loss_fn = keras.losses.SparseCategoricalCrossentropy(from_logits=False)
clf_best_path_ft = os.path.join(OUT_DIR, "clf_finetune_best.weights.h5")

print("Phase 2b: Fine-tuning ALL encoder blocks with discriminative LR...\n")
clf_ft_start = time.time()

best_val_acc = 0.0
patience_counter = 0
PATIENCE = 15
current_lr_factor = 1.0
lr_wait = 0
LR_PATIENCE = 4
prev_val_loss = float('inf')
ft_history = {"accuracy": [], "loss": [], "val_accuracy": [], "val_loss": []}

for epoch in range(CLF_EPOCHS_FINETUNE):
    epoch_loss, epoch_correct, epoch_total = [], 0, 0
    for batch_x, batch_y in train_ds_clf:
        with tf.GradientTape() as tape:
            preds = clf_model(batch_x, training=True)
            loss = loss_fn(batch_y, preds)
        grads = tape.gradient(loss, clf_model.trainable_variables)
        scaled_grads = []
        for g, v in zip(grads, clf_model.trainable_variables):
            if g is not None:
                scaled_grads.append(g * var_multipliers.get(id(v), 1.0))
            else:
                scaled_grads.append(g)
        optimizer_ft.apply_gradients(zip(scaled_grads, clf_model.trainable_variables))
        epoch_loss.append(float(loss))
        epoch_correct += int(tf.reduce_sum(tf.cast(tf.argmax(preds, axis=1) == tf.cast(batch_y, tf.int64), tf.int32)))
        epoch_total += int(batch_x.shape[0])

    train_loss, train_acc = np.mean(epoch_loss), epoch_correct / epoch_total

    val_loss_list, val_correct, val_total = [], 0, 0
    for batch_x, batch_y in val_ds_clf:
        preds = clf_model(batch_x, training=False)
        val_loss_list.append(float(loss_fn(batch_y, preds)))
        val_correct += int(tf.reduce_sum(tf.cast(tf.argmax(preds, axis=1) == tf.cast(batch_y, tf.int64), tf.int32)))
        val_total += int(batch_x.shape[0])
    val_loss, val_acc = np.mean(val_loss_list), val_correct / val_total

    ft_history["accuracy"].append(train_acc)
    ft_history["loss"].append(train_loss)
    ft_history["val_accuracy"].append(val_acc)
    ft_history["val_loss"].append(val_loss)

    improved = ""
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        clf_model.save_weights(clf_best_path_ft)
        patience_counter = 0
        improved = " << saved"
    else:
        patience_counter += 1

    if val_loss >= prev_val_loss:
        lr_wait += 1
        if lr_wait >= LR_PATIENCE:
            current_lr_factor *= 0.5
            new_lr = max(base_lr * current_lr_factor, 1e-7)
            optimizer_ft.learning_rate.assign(new_lr)
            lr_wait = 0
            print(f"  >> LR reduced to {new_lr:.1e}")
    else:
        lr_wait = 0
    prev_val_loss = val_loss

    print(f"Epoch {epoch+1:2d}/{CLF_EPOCHS_FINETUNE} — loss: {train_loss:.4f}  acc: {train_acc:.4f}  val_loss: {val_loss:.4f}  val_acc: {val_acc:.4f}{improved}")

    if patience_counter >= PATIENCE:
        print(f"\nEarly stopping at epoch {epoch+1}")
        break

clf_model.load_weights(clf_best_path_ft)
clf_ft_time = time.time() - clf_ft_start
print(f"\nPhase 2b time: {clf_ft_time:.1f}s  Best val acc: {best_val_acc:.4f}")

class FakeHistory:
    def __init__(self, h): self.history = h
clf_history_ft = FakeHistory(ft_history)
clf_train_time = clf_frozen_time + clf_ft_time

### Phase 2 — Combined Accuracy & Loss Curves

In [ ]:
combined_acc = clf_history_frozen.history["accuracy"] + clf_history_ft.history["accuracy"]
combined_val_acc = clf_history_frozen.history["val_accuracy"] + clf_history_ft.history["val_accuracy"]
combined_loss = clf_history_frozen.history["loss"] + clf_history_ft.history["loss"]
combined_val_loss = clf_history_frozen.history["val_loss"] + clf_history_ft.history["val_loss"]

phase2a_epochs = len(clf_history_frozen.history["accuracy"])

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(combined_acc, label="Train Acc")
ax1.plot(combined_val_acc, label="Val Acc")
ax1.axvline(x=phase2a_epochs - 0.5, color='gray', linestyle='--', alpha=0.7, label='Unfreeze')
ax1.set_title("AE Classifier — Accuracy")
ax1.set_xlabel("Epoch"); ax1.set_ylabel("Accuracy")
ax1.legend(); ax1.grid(True)

ax2.plot(combined_loss, label="Train Loss")
ax2.plot(combined_val_loss, label="Val Loss")
ax2.axvline(x=phase2a_epochs - 0.5, color='gray', linestyle='--', alpha=0.7, label='Unfreeze')
ax2.set_title("AE Classifier — Loss")
ax2.set_xlabel("Epoch"); ax2.set_ylabel("Loss")
ax2.legend(); ax2.grid(True)

plt.suptitle("Phase 2 — Classification (Frozen -> Fine-Tune)", fontsize=13)
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "clf_accuracy_loss.png"), dpi=150)
plt.show()

---
## 8 — Final Evaluation on Held-Out TEST Set

This is the only place we touch the test set. These are the numbers reported in the FYP.

In [ ]:
from sklearn.metrics import top_k_accuracy_score

clf_model.load_weights(clf_best_path_ft)
print("Best fine-tuned classifier weights loaded.\n")

# Re-extract embeddings with fine-tuned encoder
print("Re-extracting embeddings with fine-tuned encoder...")
ft_encoder = Model(clf_model.input, clf_model.get_layer("latent_relu").output, name="FT_Encoder")
train_embeddings_ft = ft_encoder.predict(train_ds_clf, verbose=1)
test_embeddings_ft  = ft_encoder.predict(test_ds_clf, verbose=1)

# KNN on fine-tuned embeddings
knn_ft_results = {}
for k in [1, 3, 5, 7]:
    knn = KNeighborsClassifier(n_neighbors=k, metric="cosine", n_jobs=-1)
    knn.fit(train_embeddings_ft, train_labels_arr)
    knn_preds = knn.predict(test_embeddings_ft)
    knn_acc = accuracy_score(test_labels_arr, knn_preds)
    knn_ft_results[k] = knn_acc
    print(f"KNN (k={k}, cosine, fine-tuned) test accuracy: {knn_acc*100:.2f}%")

best_knn_ft_k = max(knn_ft_results, key=knn_ft_results.get)
best_knn_ft_acc = knn_ft_results[best_knn_ft_k]

# Softmax head evaluation
test_probs = clf_model.predict(test_ds_clf, verbose=1)
test_preds = np.argmax(test_probs, axis=1)
test_true  = np.array(test_labels)

test_top1 = np.mean(test_preds == test_true)
test_top5 = top_k_accuracy_score(test_true, test_probs, k=5, labels=range(NUM_CLASSES))

print(f"\n{'='*50}")
print(f"  TEST SET RESULTS (FINAL)")
print(f"{'='*50}")
print(f"  Top-1 Accuracy:               {test_top1*100:.2f}%")
print(f"  Top-5 Accuracy:               {test_top5*100:.2f}%")
print(f"  KNN (k={best_knn_k}, pretrained):   {best_knn_acc*100:.2f}%")
print(f"  KNN (k={best_knn_ft_k}, fine-tuned):  {best_knn_ft_acc*100:.2f}%")
print(f"{'='*50}")

### 8b: Multi-Classifier Evaluation on Frozen Embeddings

In [ ]:
from sklearn.svm import SVC, LinearSVC
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import normalize
from sklearn.metrics import classification_report
from scipy.spatial.distance import cdist

train_emb_norm = normalize(train_embeddings_ft, norm="l2")
test_emb_norm  = normalize(test_embeddings_ft, norm="l2")

# KNN
best_knn_name = f"KNN (k={best_knn_ft_k}, cosine)"
best_knn_test = best_knn_ft_acc

# SVM Linear
print("Training SVM (Linear)...")
svm_linear = LinearSVC(C=1.0, max_iter=5000, random_state=SEED)
svm_linear.fit(train_emb_norm, train_labels_arr)
svm_linear_acc = accuracy_score(test_labels_arr, svm_linear.predict(test_emb_norm))
print(f"  SVM (Linear) test accuracy: {svm_linear_acc*100:.2f}%")

# SVM RBF
print("Training SVM (RBF)...")
svm_rbf = SVC(kernel="rbf", C=10.0, gamma="scale", random_state=SEED)
svm_rbf.fit(train_emb_norm, train_labels_arr)
svm_rbf_acc = accuracy_score(test_labels_arr, svm_rbf.predict(test_emb_norm))
print(f"  SVM (RBF) test accuracy:    {svm_rbf_acc*100:.2f}%")

# Logistic Regression
print("Training Logistic Regression...")
logreg = LogisticRegression(C=1.0, max_iter=2000, solver="lbfgs",
                            multi_class="multinomial", random_state=SEED)
logreg.fit(train_emb_norm, train_labels_arr)
logreg_acc = accuracy_score(test_labels_arr, logreg.predict(test_emb_norm))
print(f"  Logistic Regression test accuracy: {logreg_acc*100:.2f}%")

# Centroid Matching
print("Computing Centroid (prototype) matching...")
centroids = np.zeros((NUM_CLASSES, train_emb_norm.shape[1]))
for c in range(NUM_CLASSES):
    mask = train_labels_arr == c
    centroids[c] = np.mean(train_emb_norm[mask], axis=0)
centroids = normalize(centroids, norm="l2")
dists = cdist(test_emb_norm, centroids, metric="cosine")
centroid_preds = np.argmin(dists, axis=1)
centroid_acc = accuracy_score(test_labels_arr, centroid_preds)
print(f"  Centroid (cosine) test accuracy: {centroid_acc*100:.2f}%")

# Summary
multi_clf_results = {
    "KNN (cosine)": best_knn_test,
    "SVM (Linear)": svm_linear_acc,
    "SVM (RBF)": svm_rbf_acc,
    "Logistic Regression": logreg_acc,
    "Centroid (cosine)": centroid_acc,
    "Softmax Head (Top-1)": test_top1,
}

print(f"\n{'='*60}")
print(f"  MULTI-CLASSIFIER EVALUATION — Fine-Tuned Encoder Embeddings")
print(f"{'='*60}")
for name, acc in sorted(multi_clf_results.items(), key=lambda x: x[1], reverse=True):
    star = " *" if acc == max(multi_clf_results.values()) else ""
    print(f"  {name:28s} {acc*100:>8.2f}%{star}")
print(f"{'='*60}")
print("\n-> These lightweight classifiers run on CPU in milliseconds.")
print("-> Adding a new identity only requires a single encoder forward pass.")
print("-> No GPU retraining needed — this is the AE approach advantage.")

In [ ]:
# Bar chart
sorted_results = sorted(multi_clf_results.items(), key=lambda x: x[1], reverse=True)
names = [r[0] for r in sorted_results]
accs  = [r[1] * 100 for r in sorted_results]

colors = ['#2ecc71' if a == max(accs) else '#3498db' for a in accs]
fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.barh(names[::-1], accs[::-1], color=colors[::-1])
ax.set_xlabel("Test Accuracy (%)")
ax.set_title("Multi-Classifier Evaluation on Fine-Tuned Encoder Embeddings")
for bar, acc in zip(bars, accs[::-1]):
    ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2, f"{acc:.1f}%",
            va='center', fontsize=10)
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "multi_classifier_comparison.png"), dpi=150)
plt.show()

### 8c: Sample Predictions (test set)

In [ ]:
idxs = np.random.choice(len(test_paths), size=8, replace=False)

imgs = []
true_ids = []
for i in idxs:
    fp = test_paths[i]
    y  = test_labels[i]
    img = tf.io.read_file(fp)
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.image.resize(img, (IMG_SIZE, IMG_SIZE))
    img = tf.cast(img, tf.float32) / 255.0
    imgs.append(img)
    true_ids.append(y)

imgs_batch = tf.stack(imgs)
preds_batch = clf_model.predict(imgs_batch, verbose=0)
pred_ids = np.argmax(preds_batch, axis=1)

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()
for i in range(8):
    axes[i].imshow(imgs[i].numpy())
    true_name = class_names[true_ids[i]]
    pred_name = class_names[pred_ids[i]]
    color = 'green' if true_ids[i] == pred_ids[i] else 'red'
    axes[i].set_title(f"True: {true_name}\nPred: {pred_name}", fontsize=8, color=color)
    axes[i].axis('off')

plt.suptitle("AE Classifier — Test Set Predictions", fontsize=13)
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "sample_predictions.png"), dpi=150)
plt.show()

---
## 9 — Inference Latency Benchmark

In [ ]:
dummy = tf.random.uniform((1, IMG_SIZE, IMG_SIZE, 3))
for _ in range(10):
    _ = clf_model(dummy, training=False)

N_RUNS = 100
start = time.time()
for _ in range(N_RUNS):
    _ = clf_model(dummy, training=False)
single_latency_ms = (time.time() - start) / N_RUNS * 1000

dummy_batch = tf.random.uniform((BATCH_SIZE, IMG_SIZE, IMG_SIZE, 3))
for _ in range(10):
    _ = clf_model(dummy_batch, training=False)

start = time.time()
for _ in range(10):
    _ = clf_model(dummy_batch, training=False)
batch_latency_ms = (time.time() - start) / 10 * 1000

print(f"Inference Latency:")
print(f"  Single-image: {single_latency_ms:.2f} ms")
print(f"  Batch ({BATCH_SIZE}):   {batch_latency_ms:.2f} ms")
print(f"  Per-image (batch): {batch_latency_ms/BATCH_SIZE:.2f} ms")

---
## 10 — Latent Space Visualisation (t-SNE)

In [ ]:
from sklearn.manifold import TSNE

top10_classes = [c for c, _ in Counter(test_labels).most_common(10)]
mask = np.isin(test_true, top10_classes)

tsne = TSNE(n_components=2, perplexity=30, random_state=SEED, n_iter=1000)
coords = tsne.fit_transform(test_embeddings_ft[mask])
labels_subset = test_true[mask]

plt.figure(figsize=(10, 8))
for c in top10_classes:
    idx = labels_subset == c
    plt.scatter(coords[idx, 0], coords[idx, 1], label=class_names[c], alpha=0.7, s=30)

plt.legend(fontsize=8, loc='upper right')
plt.title(f"t-SNE of Fine-Tuned Encoder Embeddings — Test Set (top-10 classes, latent_dim={LATENT_DIM})")
plt.xlabel("t-SNE 1"); plt.ylabel("t-SNE 2")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "tsne_embeddings.png"), dpi=150)
plt.show()

---
## 11 — Save Everything

In [ ]:
# Save combined history
combined_history = {
    "frozen_" + k: [float(v) for v in vals]
    for k, vals in clf_history_frozen.history.items()
}
combined_history.update({
    "finetune_" + k: [float(v) for v in vals]
    for k, vals in clf_history_ft.history.items()
})
with open(os.path.join(OUT_DIR, "clf_history.json"), "w") as f:
    json.dump(combined_history, f, indent=2)

# Save encoder separately
encoder_save_path = os.path.join(OUT_DIR, "encoder_final.keras")
ft_encoder.save(encoder_save_path)
print(f"Encoder saved to {encoder_save_path}")

# Save full classifier
clf_save_path = os.path.join(OUT_DIR, "ae_classifier_final.keras")
clf_model.save(clf_save_path)
print(f"Classifier saved to {clf_save_path}")

# Save metrics JSON
metrics = {
    'model_name': 'AE_Classifier_Joint_Final',
    'num_classes': NUM_CLASSES,
    'img_size': IMG_SIZE,
    'batch_size': BATCH_SIZE,
    'latent_dim': LATENT_DIM,
    'phase1': {
        'type': 'joint_reconstruction_classification',
        'recon_weight': RECON_WEIGHT,
        'clf_weight': CLF_WEIGHT,
        'ae_data_size': len(ae_all_paths),
        'training_time_seconds': ae_train_time,
    },
    'phase2': {
        'frozen_epochs': len(clf_history_frozen.history['accuracy']),
        'finetune_epochs': len(clf_history_ft.history['accuracy']),
        'frozen_time_seconds': clf_frozen_time,
        'finetune_time_seconds': clf_ft_time,
        'total_clf_time_seconds': clf_train_time,
    },
    'total_training_time_seconds': ae_train_time + clf_train_time,
    'model_params': {
        'encoder_trainable': enc_train_p,
        'encoder_total': enc_train_p + enc_nontrain_p,
        'encoder_size_mb': (enc_train_p + enc_nontrain_p) * 4 / 1e6,
        'autoencoder_total': ae_train_p + ae_nontrain_p,
    },
    'dataset_sizes': {
        'train': len(train_paths),
        'val': len(val_paths),
        'test': len(test_paths),
    },
    'test_metrics': {
        'top1_accuracy': float(test_top1),
        'top5_accuracy': float(test_top5),
        'knn_pretrained': {f'k={k}': float(v) for k, v in knn_results_pretrained.items()},
        'knn_finetuned': {f'k={k}': float(v) for k, v in knn_ft_results.items()},
        'multi_classifier': {k: float(v) for k, v in multi_clf_results.items()},
    },
    'inference_latency': {
        'single_image_ms': float(single_latency_ms),
        'batch_ms': float(batch_latency_ms),
        'per_image_batch_ms': float(batch_latency_ms / BATCH_SIZE),
    }
}

with open(os.path.join(OUT_DIR, 'metrics.json'), 'w') as f:
    json.dump(metrics, f, indent=2)
print(f"Metrics saved to {os.path.join(OUT_DIR, 'metrics.json')}")

print(f"\nAll outputs saved to {OUT_DIR}")

---
## 12 — Final Summary

In [ ]:
print("="*60)
print("AUTOENCODER PIPELINE — FINAL SUMMARY (JOINT TRAINING)")
print("="*60)
print(f"")
print(f"Phase 1 (Joint Reconstruction + Classification):")
print(f"  Data:                {len(ae_all_paths)} images (max {MAX_PER_PERSON}/person)")
print(f"  AE train / val:      {len(ae_train_paths)} / {len(ae_val_paths)}")
print(f"  Loss:                {RECON_WEIGHT}*SSIM_MSE + {CLF_WEIGHT}*CrossEntropy")
print(f"  Training time:       {ae_train_time:.1f}s ({ae_train_time/60:.1f} min)")
print(f"")
print(f"Phase 2a (Head only, encoder frozen):")
print(f"  Training time:       {clf_frozen_time:.1f}s")
print(f"  Best val accuracy:   {max(clf_history_frozen.history['val_accuracy']):.4f}")
print(f"Phase 2b (Fine-tune, discriminative LR):")
print(f"  Training time:       {clf_ft_time:.1f}s")
print(f"  Best val accuracy:   {best_val_acc:.4f}")
print(f"")
print(f"Classification Data:")
print(f"  Identities:          {NUM_CLASSES}")
print(f"  Train / Val / Test:  {len(train_paths)} / {len(val_paths)} / {len(test_paths)}")
print(f"")
print(f"Model:")
print(f"  Latent dimension:    {LATENT_DIM}")
print(f"  Encoder params:      {enc_train_p+enc_nontrain_p:,}")
print(f"  Encoder size:        {(enc_train_p+enc_nontrain_p)*4/1e6:.2f} MB")
print(f"  Total train time:    {ae_train_time + clf_train_time:.1f}s")
print(f"")
print(f"TEST SET RESULTS (reported in FYP):")
print(f"  Softmax Head:")
print(f"    Top-1 accuracy:    {test_top1*100:.2f}%")
print(f"    Top-5 accuracy:    {test_top5*100:.2f}%")
print(f"")
print(f"  Multi-Classifier on Encoder Embeddings:")
for name, acc in sorted(multi_clf_results.items(), key=lambda x: x[1], reverse=True):
    print(f"    {name:28s} {acc*100:.2f}%")
print(f"")
print(f"  KNN (pretrained, k={best_knn_k}):   {best_knn_acc*100:.2f}%")
print(f"")
print(f"Inference:")
print(f"  Single-image:        {single_latency_ms:.2f} ms")
print(f"  Batch ({BATCH_SIZE}):          {batch_latency_ms:.2f} ms")
print("="*60)